# DoTA-CAP frame archive — stream the CAP tars **once**, keep one zip per DoTA clip (CPU)

`core/tools/dota_cap_frames.py`. For each of the **1,129** clips of the frozen DoTA-CAP alignment (dev 569 + eval 560),
`pack` writes `{dota_id}.zip` = the clip's **DoTA-native frame sequence** rebuilt by its frozen frame map, with the
**original CAP JPEG bytes**, stored and not re-encoded. That is exactly what every earlier DoTA-CAP tool rebuilt from the
stream, so caches built from the archive equal caches built from the tars. Entry count = `DoTA_s1_ncc` rows (s1).

**Runtime: CPU** (no model). Needs ≥ 24 GB free on `/content` (3 × the 8 GB stream batch). Resumable: re-run step 2
after a disconnect, finished clips are skipped (a torn zip counts as not done).

Eval clips are only **written** here, never read or scored. DoTA-CAP-eval stays a final-only set for every read-out.

| step | what |
|---|---|
| 1 | setup |
| 2 | **pack**: per CAP group, stream its tar parts and zip each DoTA clip it contains (stops early per group) |
| 3 | **verify**: every aligned clip has a zip whose entry count = frame map = `DoTA_s1_ncc` rows → writes `frames_manifest.json` |
| 4 | before deleting the tar parts: read the note |

Output: `Thesis-Final/data/DoTA_CAP_frames/` (sizes printed by step 3).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: CAP tar parts (data/MMAU/raw), DoTA_s1_ncc
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: the frozen alignment
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + the archive
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'
os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core/tools/dota_cap_frames.py').is_file(), f'no dota_cap_frames.py under {REPO} -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]
from core import constants  # noqa: E402

RAW       = constants.DATA_ROOT / 'MMAU' / 'raw'
GROUPS    = ['1-10', '11', '12-42', '43', '44-62']                    # CAP-DATA tar groups
ALIGNMENT = Path(DRIVE_V2) / 'outputs' / 'REPORTS' / 'dota_cap' / 'dota_cap_alignment.json'
DOTA_S1   = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
ARCHIVE   = Path(DRIVE_F) / 'data' / constants.DOTA_CAP_FRAMES_DIRNAME
REPORTS   = Path(DRIVE_F) / 'outputs' / 'REPORTS' / 'dota_cap_frames'
LOCAL     = Path('/content/pack')
BATCH_GB  = constants.MMAU_STREAM_BATCH_GB
for p in (ARCHIVE, REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name, path in (('alignment', ALIGNMENT), ('DoTA_s1_ncc', DOTA_S1), ('CAP annotations', RAW / 'cap_text_annotations.xls')):
    assert path.exists(), f'missing {name}: {path}'
for g in GROUPS:
    assert sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*')), f'no CAP parts for group {g}'
free_gb = shutil.disk_usage('/content').free / 1e9
assert free_gb >= 3 * BATCH_GB, f'{free_gb:.0f} GB free on /content, need {3 * BATCH_GB:.0f}'
print('archive ->', ARCHIVE, f'| /content free {free_gb:.0f} GB')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av xlrd
python -c "import av, xlrd; print('deps OK')"

In [ ]:
import pandas as pd

from core.tools import dota_cap_frames as dcf

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd, log=None):
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}')


clips = dcf.aligned_clips(ALIGNMENT)
assert len(clips) == 1129, len(clips)
ann = pd.read_excel(RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
done = {k for k, c in clips.items() if dcf.archive_frames(dcf.archive_path(ARCHIVE, k)) == c['dota_frames']}
print(f'{len(done)} / {len(clips)} clips already packed')

## 2. Pack (CPU, resumable). **Re-run this cell after a disconnect.**

Each group's tar is read as one stream and stops as soon as the last CAP video needed from that group is closed.

In [ ]:
for g in GROUPS:
    lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
    need = sorted({c['cap_id'] for k, c in clips.items()
                   if lo <= type_of[c['cap_id']] <= hi
                   and dcf.archive_frames(dcf.archive_path(ARCHIVE, k)) != c['dota_frames']})
    print(f'group {g}: {len(need)} CAP videos to stream')
    if not need:
        continue
    ids_file = LOCAL / f'cap_ids_{g}.txt'
    ids_file.write_text(''.join(f'{i}\n' for i in need))
    parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.dota_cap_frames', 'pack', '--alignment', ALIGNMENT, '--parts', *parts,
         '--work-dir', LOCAL / 'work' / g, '--cap-ids-file', ids_file, '--archive-dir', ARCHIVE,
         '--max-batch-gb', BATCH_GB, '--report', REPORTS / f'pack_{g}.json'], log=LOCAL / 'pack.log')
    print(f'group {g}: {(time.time() - t0) / 60:.0f} min')

## 3. Verify → `frames_manifest.json` (needed by `clip_windows archive`)

In [ ]:
run([sys.executable, '-m', 'core.tools.dota_cap_frames', 'verify', '--alignment', ALIGNMENT, '--archive-dir', ARCHIVE,
     '--dota-clip-dir', DOTA_S1, '--report', REPORTS / 'verify.json'])
v = json.loads((REPORTS / 'verify.json').read_text())
print({k: v[k] for k in ('clips', 'ok', 'frames', 'gb', 'pass')})
assert v['pass'], f"{len(v['failed'])} clips incomplete: {dict(list(v['failed'].items())[:10])}"
drive.flush_and_unmount()
print('archive complete and flushed:', ARCHIVE)

## 4. Before deleting `Thesis/data/MMAU/raw/CAP-DATA_chunks/`

This notebook deliberately **does not delete anything**. Before deleting the parts by hand:

- The archive holds only the **1,129 DoTA-CAP clips**, at DoTA's 10 fps frame sequence. The tar parts are **all of
  MM-AU's CAP-DATA** (~11.7 K videos, 30 fps). The proposal keeps MM-AU as an optional zero-shot test set (§4.1), and
  `colab/v2/mmau_p0.ipynb` reads these parts. Deleting them closes MM-AU unless you can download it again.
- CAP's native 30 fps frames between DoTA frames are not kept.
- Delete only after step 3 printed `pass: True`, and after one `clip_windows archive` run has passed its 1×1 corpus
  gate (STP notebook step 5).